# Stage C 03c — paper-deep memory mathematical oracle

Run this first on a Mac CPU or Colab CPU. It validates the FP64 equations, causal projection history, checkpoint format, and exact accelerated/reference parity. It does not train the genomic model.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='2cf1e9fc87025a71b10121ba9452c8175ec0fbb3'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'


In [ ]:
from pathlib import Path
import json, subprocess, sys
IN_COLAB='google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    mount=Path('/content/drive')
    if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
    repo=Path('/content/SeqTrainer')
else:
    repo=Path.cwd()
if not (repo/'.git').exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan,dev]'],check=True)
commit=subprocess.run(['git','-C',str(repo),'rev-parse','HEAD'],check=True,capture_output=True,text=True).stdout.strip()
print('Testing exact source commit:',commit)


In [ ]:
if IN_COLAB:
    out=Path(DRIVE_ROOT)/'runs'/'c12_paper_deep_fp64_oracle'
else:
    out=repo/'artifacts'/'c12_paper_deep_fp64_oracle'
out.mkdir(parents=True,exist_ok=True)
tests=[
 str(repo/'tests/test_titans_paper_mac_memory.py'),
 str(repo/'tests/test_titans_paper_mac_stage_b_exact_acceleration.py'),
 str(repo/'tests/test_titans_paper_mac_stage_c_model.py'),
]
command=[sys.executable,'-m','pytest','-q',*tests]
result=subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(out),'--label','paper_deep_fp64_oracle','--repo',str(repo),'--',*command])
log=out/'logs/paper_deep_fp64_oracle.log'
if log.exists(): print(log.read_text(errors='replace')[-16000:])
if result.returncode: raise RuntimeError('Paper-deep FP64 oracle failed; do not proceed to 03d.')
report={'format_version':1,'passed':True,'code_commit':commit,'tests':tests,'log':str(log)}
(out/'oracle_report.json').write_text(json.dumps(report,indent=2,sort_keys=True)+'\n')
print('PASS. Next notebook: 03d_stage_c_paper_deep_memory_t4_stability.ipynb')
print('Evidence:',out)
